# Proyecto Final — Detección de Cáncer en Imágenes Histopatológicas

**Competencia:** Histopathologic Cancer Detection — Kaggle  
**Link:** https://www.kaggle.com/competitions/histopathologic-cancer-detection  
**Problema:** Clasificación binaria de imágenes histopatológicas: presencia (`label=1`) o ausencia (`label=0`) de tejido tumoral.

Este notebook cubre lo pedido en el entregable:

1. Objetivo de negocio, población y target.  
2. Lectura de información.  
3. Tratamiento de la información.  
4. Transformación de imágenes a tensores.  
5. Entrenamiento de red neuronal base.  
6. Entrenamiento de **3 arquitecturas diferentes**.  
7. Selección de red final y optimización por umbral.  
8. Análisis de resultados.  
9. Recomendaciones de negocio.

Arquitecturas evaluadas:

- **CNN Base**: baseline convolucional propia.
- **EfficientNetB0**: CNN moderna con transfer learning.
- **MLP-Mixer**: arquitectura basada en patches y capas MLP, alternativa no convolucional clásica.

## 0. Objetivo de negocio, población y target

### Objetivo de negocio

Desarrollar un modelo de deep learning que ayude a priorizar imágenes histopatológicas sospechosas de cáncer, permitiendo apoyar el flujo de revisión médica mediante una predicción automática de probabilidad de tejido tumoral.

### ¿Cómo se soluciona?

Se entrenarán modelos de clasificación binaria sobre imágenes histopatológicas. Cada modelo recibirá una imagen como entrada y devolverá una probabilidad entre 0 y 1:

- `0`: no se detecta tejido tumoral.
- `1`: se detecta tejido tumoral.

### Población

Imágenes histopatológicas digitales provenientes del dataset de la competencia **Histopathologic Cancer Detection**.

### Target

La variable objetivo es `label`:

- `label = 0`: imagen negativa.
- `label = 1`: imagen positiva para tejido tumoral.

### Criterio de éxito

Se prioriza principalmente:

- **AUC-ROC**: capacidad general de separación entre clases.
- **Recall/Sensibilidad**: importante porque en cáncer es crítico reducir falsos negativos.
- **F1-score**: balance entre precision y recall.

> Nota ética: este modelo no reemplaza a un médico. Debe entenderse como una herramienta de apoyo y priorización.

In [ ]:
# ===============================================================
# PASO 1: IMPORTS, SEMILLAS Y CONFIGURACIÓN GENERAL
# ===============================================================

import os
import random
import shutil
import zipfile
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    ConfusionMatrixDisplay,
)
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU disponible:", tf.config.list_physical_devices("GPU"))

In [ ]:
# ===============================================================
# PASO 2: MONTAR GOOGLE DRIVE Y DEFINIR RUTAS
# ===============================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# Cambia esta ruta si tu ZIP está en otra carpeta de Drive.
zip_drive = "/content/drive/MyDrive/Deep_learning/histopathologic-cancer-detection.zip"
zip_local = "/content/histopathologic-cancer-detection.zip"
BASE_DIR = "/content/drive/MyDrive/dataset"

COMPETITION_URL = "https://www.kaggle.com/competitions/histopathologic-cancer-detection"

print("Competencia:", COMPETITION_URL)
print("¿Existe el ZIP en Drive?:", os.path.exists(zip_drive))

if os.path.exists(zip_drive):
    print("Tamaño ZIP en GB:", round(os.path.getsize(zip_drive) / (1024**3), 2))
else:
    raise FileNotFoundError(f"No se encontró el ZIP en: {zip_drive}")

In [ ]:
# ===============================================================
# PASO 3: COPIAR ZIP A /content Y DESCOMPRIMIR
# ===============================================================

os.makedirs(BASE_DIR, exist_ok=True)

if not os.path.exists(zip_local):
    print("Copiando ZIP desde Drive hacia /content...")
    shutil.copy2(zip_drive, zip_local)
    print("ZIP copiado correctamente.")
else:
    print("El ZIP ya existe en /content. Saltando copia.")

train_expected = os.path.join(BASE_DIR, "train")
test_expected = os.path.join(BASE_DIR, "test")
labels_expected = os.path.join(BASE_DIR, "train_labels.csv")

if not (os.path.exists(train_expected) and os.path.exists(test_expected) and os.path.exists(labels_expected)):
    print("Descomprimiendo dataset...")
    with zipfile.ZipFile(zip_local, "r") as z:
        z.extractall(BASE_DIR)
    print("Dataset descomprimido.")
else:
    print("El dataset ya está descomprimido. Saltando extracción.")

print("Contenido de BASE_DIR:", os.listdir(BASE_DIR))

In [ ]:
# ===============================================================
# PASO 4: EXPLORACIÓN DE ESTRUCTURA Y TAMAÑO DE DATA
# ===============================================================

train_dir = os.path.join(BASE_DIR, "train")
test_dir = os.path.join(BASE_DIR, "test")
labels_path = os.path.join(BASE_DIR, "train_labels.csv")
sample_submission_path = os.path.join(BASE_DIR, "sample_submission.csv")

train_files = [f for f in os.listdir(train_dir) if f.endswith(".tif")]
test_files = [f for f in os.listdir(test_dir) if f.endswith(".tif")]

print("Train dir:", train_dir)
print("Test dir:", test_dir)
print("Labels CSV:", labels_path)
print("Sample submission:", sample_submission_path)
print("Imágenes en train:", len(train_files))
print("Imágenes en test Kaggle:", len(test_files))
print("Total de imágenes:", len(train_files) + len(test_files))

In [ ]:
# ===============================================================
# PASO 5: CARGAR CSV DE ETIQUETAS Y CREAR RUTAS DE IMÁGENES
# ===============================================================

df = pd.read_csv(labels_path)

# El CSV tiene columnas: id, label.
# La imagen correspondiente está en train/id.tif.
df["path"] = df["id"].astype(str).apply(lambda x: os.path.join(train_dir, x + ".tif"))
df["exists"] = df["path"].apply(os.path.exists)

print("Imágenes con ruta existente:", df["exists"].sum(), "/", len(df))

df = df[df["exists"]].drop(columns=["exists"]).reset_index(drop=True)

display(df.head())
print(df.info())

print("\nDistribución de clases:")
print(df["label"].value_counts())

print("\nDistribución porcentual:")
print((df["label"].value_counts(normalize=True) * 100).round(2))

In [ ]:
# ===============================================================
# PASO 6: VISUALIZACIÓN DE IMÁGENES
# ===============================================================

sample_df = df.sample(25, random_state=SEED).reset_index(drop=True)

plt.figure(figsize=(12, 12))
for i, row in sample_df.iterrows():
    img = Image.open(row["path"]).convert("RGB")
    plt.subplot(5, 5, i + 1)
    plt.imshow(img)
    plt.title(f"label={row['label']}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# ===============================================================
# PASO 7: DEMO DE IMAGEN A TENSOR
# ===============================================================

sample_path = df.loc[0, "path"]
sample_label = df.loc[0, "label"]

img_pil = Image.open(sample_path).convert("RGB")
img_arr = np.asarray(img_pil, dtype=np.float32)
img_norm = img_arr / 255.0
img_vector = img_norm.reshape(-1)

print("Ruta:", sample_path)
print("Label:", sample_label)
print("Tamaño PIL:", img_pil.size)
print("Tensor imagen:", img_norm.shape)
print("Rango normalizado:", img_norm.min(), "a", img_norm.max())
print("Vector aplanado:", img_vector.shape)

plt.figure(figsize=(4, 4))
plt.imshow(img_pil)
plt.title(f"Imagen de ejemplo - label={sample_label}")
plt.axis("off")
plt.show()

## 1. Tratamiento de la información

El dataset original de Kaggle tiene un conjunto `train` con etiquetas y un conjunto `test` sin etiquetas.  
Como el `test` de Kaggle no tiene `label`, para poder reportar métricas reales se crea un **test interno** desde `train_labels.csv`.

Se usará la siguiente división:

- 70% entrenamiento.
- 15% validación.
- 15% test interno.

El conjunto `test` oficial de Kaggle se usará solo para generar el archivo `submission.csv`.

In [ ]:
# ===============================================================
# PASO 8: TRAIN / VALIDATION / TEST INTERNO ESTRATIFICADO
# ===============================================================

# Para pruebas rápidas puedes activar esto.
# Para el entregable final, déjalo en False.
USE_SAMPLE = False
SAMPLE_PER_CLASS = 6000  # Solo se usa si USE_SAMPLE=True

work_df = df.copy()

if USE_SAMPLE:
    work_df = (
        work_df.groupby("label", group_keys=False)
        .apply(lambda x: x.sample(min(len(x), SAMPLE_PER_CLASS), random_state=SEED))
        .sample(frac=1, random_state=SEED)
        .reset_index(drop=True)
    )
    print("Modo muestra activado:", work_df.shape)

train_df, temp_df = train_test_split(
    work_df,
    test_size=0.30,
    stratify=work_df["label"],
    random_state=SEED,
)

val_df, test_internal_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=SEED,
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_internal_df = test_internal_df.reset_index(drop=True)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test interno:", test_internal_df.shape)

print("\nDistribución train:")
print(train_df["label"].value_counts(normalize=True).round(3))

print("\nDistribución validation:")
print(val_df["label"].value_counts(normalize=True).round(3))

print("\nDistribución test interno:")
print(test_internal_df["label"].value_counts(normalize=True).round(3))

## 2. Transformación de imágenes a tensores

Cada imagen `.tif` se transforma así:

1. Lectura con PIL.
2. Conversión a RGB.
3. Redimensionamiento a `96x96`.
4. Conversión a arreglo `float32`.
5. Normalización de píxeles a rango `[0, 1]`.
6. Conversión a batch usando `tf.data.Dataset`.

La salida para cada imagen será un tensor de tamaño:

```text
96 x 96 x 3
```

In [ ]:
# ===============================================================
# PASO 9: DATASET TF.DATA PARA IMÁGENES .TIF
# ===============================================================

IMG_SIZE = (96, 96)
BATCH = 64
AUTOTUNE = tf.data.AUTOTUNE

train_paths = train_df["path"].values
train_labels = train_df["label"].astype("int32").values

val_paths = val_df["path"].values
val_labels = val_df["label"].astype("int32").values

test_internal_paths = test_internal_df["path"].values
test_internal_labels = test_internal_df["label"].astype("int32").values


def load_tif_np(path, label):
    path = path.decode("utf-8")
    img = Image.open(path).convert("RGB").resize(IMG_SIZE)
    img = np.asarray(img, dtype=np.float32) / 255.0
    return img, np.int32(label)


def load_tif_tf(path, label):
    img, label = tf.numpy_function(
        load_tif_np,
        [path, label],
        [tf.float32, tf.int32]
    )
    img.set_shape((IMG_SIZE[0], IMG_SIZE[1], 3))
    label.set_shape(())
    return img, label


def make_dataset(paths, labels, training=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(buffer_size=min(len(paths), 10000), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_tif_tf, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH)
    ds = ds.prefetch(AUTOTUNE)
    return ds


train_ds = make_dataset(train_paths, train_labels, training=True)
val_ds = make_dataset(val_paths, val_labels, training=False)
test_internal_ds = make_dataset(test_internal_paths, test_internal_labels, training=False)

for images, labels in train_ds.take(1):
    print("Batch imágenes:", images.shape)
    print("Batch labels:", labels.shape)
    print("Rango pixeles:", tf.reduce_min(images).numpy(), "a", tf.reduce_max(images).numpy())

In [ ]:
# ===============================================================
# PASO 10: CLASS WEIGHTS PARA DESBALANCE DE CLASES
# ===============================================================

classes = np.array([0, 1])

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_labels,
)

class_weight = {int(cls): float(w) for cls, w in zip(classes, weights)}
print("class_weight:", class_weight)

In [ ]:
# ===============================================================
# PASO 11: DATA AUGMENTATION Y CALLBACKS
# ===============================================================

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical", seed=SEED),
    layers.RandomRotation(0.10, seed=SEED),
    layers.RandomZoom(0.10, seed=SEED),
], name="data_augmentation")


def make_callbacks(model_name, patience=3):
    return [
        keras.callbacks.EarlyStopping(
            monitor="val_auc",
            mode="max",
            patience=patience,
            restore_best_weights=True,
            verbose=1,
        ),
        keras.callbacks.ModelCheckpoint(
            filepath=f"/content/best_{model_name}.keras",
            monitor="val_auc",
            mode="max",
            save_best_only=True,
            verbose=1,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_auc",
            mode="max",
            factor=0.5,
            patience=2,
            min_lr=1e-6,
            verbose=1,
        ),
    ]


def compile_binary_model(model, lr=1e-3):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            keras.metrics.AUC(name="auc"),
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
        ],
    )
    return model

# 3. Arquitectura 1 — CNN Base

Esta es la red baseline del proyecto.  
Sirve como punto de comparación para saber si arquitecturas más avanzadas realmente mejoran el rendimiento.

In [ ]:
# ===============================================================
# PASO 12: MODELO 1 - CNN BASELINE
# ===============================================================

cnn = keras.Sequential([
    layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),
    data_augmentation,

    layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(256, (3, 3), padding="same", activation="relu"),
    layers.BatchNormalization(),

    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.40),
    layers.Dense(1, activation="sigmoid"),
], name="CNN_Base")

cnn = compile_binary_model(cnn, lr=1e-3)
cnn.summary()

In [ ]:
# ===============================================================
# PASO 13: ENTRENAMIENTO CNN BASELINE
# ===============================================================

EPOCHS_CNN = 10

history_cnn = cnn.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_CNN,
    class_weight=class_weight,
    callbacks=make_callbacks("cnn_base", patience=3),
    verbose=1,
)

# 4. Arquitectura 2 — EfficientNetB0 con Transfer Learning

EfficientNetB0 es una CNN moderna preentrenada en ImageNet.  
La idea es aprovechar características visuales ya aprendidas y adaptarlas al problema de cáncer histopatológico.

En esta primera etapa se congela la base preentrenada y se entrena solo el clasificador final.

In [ ]:
# ===============================================================
# PASO 14: MODELO 2 - EFFICIENTNETB0 TRANSFER LEARNING
# ===============================================================

base_effnet = keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3),
)

base_effnet.trainable = False

inputs = keras.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
x = data_augmentation(inputs)

# Las imágenes vienen normalizadas de 0 a 1.
# EfficientNet de Keras incluye preprocessing interno y espera una escala tipo 0 a 255.
x = layers.Rescaling(255.0)(x)

x = base_effnet(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.40)(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.30)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

effnet = keras.Model(inputs, outputs, name="EfficientNetB0_TransferLearning")
effnet = compile_binary_model(effnet, lr=1e-4)
effnet.summary()

In [ ]:
# ===============================================================
# PASO 15: ENTRENAMIENTO EFFICIENTNETB0
# ===============================================================

EPOCHS_EFFNET = 10

history_effnet = effnet.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_EFFNET,
    class_weight=class_weight,
    callbacks=make_callbacks("effnet_b0", patience=3),
    verbose=1,
)

# 5. Arquitectura 3 — MLP-Mixer

MLP-Mixer es una arquitectura basada en patches y capas densas.  
A diferencia de una CNN, no usa convoluciones para extraer características espaciales.  
Divide la imagen en parches y mezcla información de dos formas:

- **Token mixing:** mezcla información entre patches.
- **Channel mixing:** mezcla información entre canales/características.

Esta arquitectura permite comparar un enfoque no convolucional frente a CNN y transfer learning.

In [ ]:
class Patches(layers.Layer):
    def __init__(self, patch_size, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size

    def call(self, images):
        batch_size = tf.shape(images)[0]

        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding="VALID",
        )

        patch_dims = patches.shape[-1]

        num_patches = (
            images.shape[1] // self.patch_size
        ) * (
            images.shape[2] // self.patch_size
        )

        patches = tf.reshape(
            patches,
            [batch_size, num_patches, patch_dims]
        )

        return patches

    def get_config(self):
        config = super().get_config()
        config.update({"patch_size": self.patch_size})
        return config

In [ ]:
# ===============================================================
# PASO 17: MODELO 3 - MLP-MIXER
# ===============================================================

PATCH_SIZE = 8
NUM_PATCHES = (IMG_SIZE[0] // PATCH_SIZE) * (IMG_SIZE[1] // PATCH_SIZE)
PROJECTION_DIM = 64
TOKEN_MLP_DIM = 128
CHANNEL_MLP_DIM = 128
NUM_MIXER_BLOCKS = 4

inputs = keras.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
x = data_augmentation(inputs)
x = Patches(PATCH_SIZE)(x)
x = layers.Dense(PROJECTION_DIM)(x)

for i in range(NUM_MIXER_BLOCKS):
    x = MLPMixerBlock(
        num_patches=NUM_PATCHES,
        projection_dim=PROJECTION_DIM,
        token_mlp_dim=TOKEN_MLP_DIM,
        channel_mlp_dim=CHANNEL_MLP_DIM,
        dropout=0.10,
        name=f"mixer_block_{i+1}",
    )(x)

x = layers.LayerNormalization()(x)
x = layers.GlobalAveragePooling1D()(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.40)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

mlp_mixer = keras.Model(inputs, outputs, name="MLP_Mixer")
mlp_mixer = compile_binary_model(mlp_mixer, lr=1e-3)
mlp_mixer.summary()

In [ ]:
# ===============================================================
# PASO 18: ENTRENAMIENTO MLP-MIXER
# ===============================================================

EPOCHS_MLP = 10

history_mlp_mixer = mlp_mixer.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_MLP,
    class_weight=class_weight,
    callbacks=make_callbacks("mlp_mixer", patience=3),
    verbose=1,
)

# 6. Curvas de entrenamiento

Se comparan las curvas de `loss`, `accuracy` y `AUC` para observar convergencia y posible sobreajuste.

In [ ]:
# ===============================================================
# PASO 19: CURVAS DE ENTRENAMIENTO
# ===============================================================

histories = {
    "CNN Base": history_cnn,
    "EfficientNetB0": history_effnet,
    "MLP-Mixer": history_mlp_mixer,
}


def plot_history(history, metric, title_prefix=""):
    plt.figure(figsize=(8, 5))
    plt.plot(history.history[metric], label=f"train_{metric}")
    plt.plot(history.history[f"val_{metric}"], label=f"val_{metric}")
    plt.xlabel("Época")
    plt.ylabel(metric)
    plt.title(f"{title_prefix} - {metric}")
    plt.legend()
    plt.grid(True)
    plt.show()


for model_name, hist in histories.items():
    for metric in ["loss", "accuracy", "auc"]:
        plot_history(hist, metric, model_name)

# 7. Evaluación y comparación de modelos

Para la selección del modelo se usan métricas de clasificación binaria:

- Accuracy
- Precision
- Recall / sensibilidad
- F1-score
- AUC-ROC

En problemas médicos, el **recall** es especialmente importante porque mide cuántos casos positivos reales fueron detectados.

In [ ]:
# ===============================================================
# PASO 20: FUNCIONES DE EVALUACIÓN
# ===============================================================

models = {
    "CNN Base": cnn,
    "EfficientNetB0": effnet,
    "MLP-Mixer": mlp_mixer,
}


def predict_prob(model, dataset):
    return model.predict(dataset, verbose=0).ravel()


def evaluate_predictions(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1_score": f1_score(y_true, y_pred, zero_division=0),
        "auc": roc_auc_score(y_true, y_prob),
        "threshold": threshold,
    }


def evaluate_model(name, model, dataset, y_true, threshold=0.5):
    y_prob = predict_prob(model, dataset)
    result = evaluate_predictions(y_true, y_prob, threshold=threshold)
    result["modelo"] = name
    return result, y_prob


def find_best_threshold(y_true, y_prob, metric="f1"):
    thresholds = np.linspace(0.05, 0.95, 181)
    best_t = 0.5
    best_score = -1

    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        if metric == "f1":
            score = f1_score(y_true, y_pred, zero_division=0)
        elif metric == "recall":
            score = recall_score(y_true, y_pred, zero_division=0)
        else:
            raise ValueError("metric debe ser 'f1' o 'recall'")

        if score > best_score:
            best_score = score
            best_t = t

    return float(best_t), float(best_score)

In [ ]:
# ===============================================================
# PASO 21: COMPARACIÓN EN VALIDACIÓN
# ===============================================================

y_true_val = val_df["label"].astype(int).values

resultados_val = []
val_probs = {}

for name, model in models.items():
    result, prob = evaluate_model(name, model, val_ds, y_true_val, threshold=0.5)
    resultados_val.append(result)
    val_probs[name] = prob

resultados_val_df = pd.DataFrame(resultados_val)
resultados_val_df = resultados_val_df[["modelo", "accuracy", "precision", "recall", "f1_score", "auc", "threshold"]]
resultados_val_df = resultados_val_df.sort_values(by=["auc", "recall", "f1_score"], ascending=False).reset_index(drop=True)

display(resultados_val_df)

In [ ]:
# ===============================================================
# PASO 22: SELECCIÓN DE MODELO FINAL + OPTIMIZACIÓN DE UMBRAL
# ===============================================================

best_model_name = resultados_val_df.iloc[0]["modelo"]
best_model = models[best_model_name]
best_val_prob = val_probs[best_model_name]

best_threshold, best_f1_val = find_best_threshold(y_true_val, best_val_prob, metric="f1")

print("Modelo final seleccionado:", best_model_name)
print("Criterio de selección: mayor AUC, luego recall y F1 en validación")
print("Umbral optimizado por F1 en validación:", round(best_threshold, 4))
print("Mejor F1 en validación con ese umbral:", round(best_f1_val, 4))

optimized_val_metrics = evaluate_predictions(y_true_val, best_val_prob, threshold=best_threshold)
print("\nMétricas del modelo final en validación con umbral optimizado:")
for k, v in optimized_val_metrics.items():
    print(f"{k}: {v:.4f}")

# 8. Rendimiento en test interno

Ahora se evalúa el modelo final sobre el **test interno**, que no fue usado para entrenar ni para escoger el umbral.

In [ ]:
# ===============================================================
# PASO 23: EVALUACIÓN FINAL EN TEST INTERNO
# ===============================================================

y_true_test = test_internal_df["label"].astype(int).values

test_prob_final = predict_prob(best_model, test_internal_ds)
test_metrics = evaluate_predictions(y_true_test, test_prob_final, threshold=best_threshold)

resultados_test_df = pd.DataFrame([{ "modelo": best_model_name, **test_metrics }])
resultados_test_df = resultados_test_df[["modelo", "accuracy", "precision", "recall", "f1_score", "auc", "threshold"]]

display(resultados_test_df)

print("Reporte de clasificación en test interno:")
y_pred_test = (test_prob_final >= best_threshold).astype(int)
print(classification_report(y_true_test, y_pred_test, digits=4, zero_division=0))

In [ ]:
# ===============================================================
# PASO 24: MATRIZ DE CONFUSIÓN Y ROC DEL MODELO FINAL
# ===============================================================

cm = confusion_matrix(y_true_test, y_pred_test)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No cáncer", "Cáncer"])
disp.plot(values_format="d")
plt.title(f"Matriz de confusión - {best_model_name}")
plt.grid(False)
plt.show()

fpr, tpr, _ = roc_curve(y_true_test, test_prob_final)
auc_test = roc_auc_score(y_true_test, test_prob_final)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"AUC = {auc_test:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate / Recall")
plt.title(f"Curva ROC - {best_model_name}")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# ===============================================================
# PASO 25: VISUALIZACIÓN DE PREDICCIONES DEL MODELO FINAL
# ===============================================================

examples = test_internal_df.sample(16, random_state=SEED).reset_index(drop=True)
example_paths = examples["path"].values
example_labels = examples["label"].astype(int).values

example_ds = make_dataset(example_paths, example_labels, training=False)
example_probs = predict_prob(best_model, example_ds)
example_preds = (example_probs >= best_threshold).astype(int)

plt.figure(figsize=(12, 12))
for i, row in examples.iterrows():
    img = Image.open(row["path"]).convert("RGB")
    plt.subplot(4, 4, i + 1)
    plt.imshow(img)
    plt.title(
        f"Real={row['label']} | Pred={example_preds[i]}\nProb={example_probs[i]:.3f}",
        fontsize=9,
    )
    plt.axis("off")

plt.tight_layout()
plt.show()

# 9. Análisis de resultados

Interpretación sugerida para el informe:

1. La **CNN Base** funciona como baseline y permite validar que el pipeline de datos, tensores y entrenamiento es correcto.
2. **EfficientNetB0** aprovecha transfer learning, por lo que suele aprender representaciones visuales más robustas con menos entrenamiento.
3. **MLP-Mixer** permite evaluar una arquitectura distinta, basada en patches y capas densas, sin depender directamente de convoluciones.
4. La red final se selecciona priorizando **AUC** y **recall**, ya que en un contexto médico es importante detectar la mayor cantidad posible de casos positivos.
5. El umbral de decisión se optimiza con validación para mejorar F1-score, en lugar de usar siempre 0.5.

In [ ]:
# ===============================================================
# PASO 26: RESUMEN AUTOMÁTICO PARA INFORME/PPT
# ===============================================================

summary = {
    "competencia": COMPETITION_URL,
    "imagenes_train_original": len(train_files),
    "imagenes_test_kaggle": len(test_files),
    "train_usado": len(train_df),
    "validacion_usada": len(val_df),
    "test_interno_usado": len(test_internal_df),
    "modelo_final": best_model_name,
    "umbral_optimizado": best_threshold,
    "metricas_validacion": optimized_val_metrics,
    "metricas_test_interno": test_metrics,
}

print(json.dumps(summary, indent=4))

print("\nTexto sugerido para PPT:")
texto_ppt = (
    "Se evaluaron tres arquitecturas: CNN Base, EfficientNetB0 con transfer learning y MLP-Mixer.\n"
    f"El modelo seleccionado fue {best_model_name}, debido a su mejor desempeño en validación considerando AUC, recall y F1-score.\n"
    f"Luego se optimizó el umbral de decisión a {best_threshold:.3f} para mejorar el balance entre falsos positivos y falsos negativos.\n"
    f"En el test interno, el modelo obtuvo AUC={test_metrics['auc']:.4f}, recall={test_metrics['recall']:.4f} y F1-score={test_metrics['f1_score']:.4f}."
)
print(texto_ppt)

# 10. Recomendaciones de negocio

La solución puede ayudar al negocio de las siguientes formas:

1. **Priorización de casos sospechosos:** las imágenes con mayor probabilidad de cáncer pueden revisarse primero.
2. **Apoyo al especialista:** el modelo puede funcionar como segunda lectura, no como reemplazo médico.
3. **Reducción de carga operativa:** permite filtrar grandes volúmenes de imágenes y enfocar la revisión humana.
4. **Control de riesgo:** se recomienda priorizar recall/sensibilidad para reducir falsos negativos.
5. **Uso recomendado de la salida del modelo:** mostrar la probabilidad de cáncer y clasificar casos en niveles de riesgo:
   - Bajo riesgo: probabilidad baja.
   - Riesgo medio: requiere revisión.
   - Alto riesgo: revisión prioritaria.

Para despliegue real, se recomienda validar el modelo con datos externos, auditar sesgos y mantener revisión médica obligatoria.

In [ ]:
# ===============================================================
# PASO 27: GUARDAR MODELO FINAL, MÉTRICAS Y RESULTADOS
# ===============================================================

output_dir = "/content/drive/MyDrive/Deep_learning/resultados_proyecto_cancer"
os.makedirs(output_dir, exist_ok=True)

safe_model_name = best_model_name.lower().replace(" ", "_").replace("-", "_")
model_drive_path = os.path.join(output_dir, f"modelo_final_{safe_model_name}.keras")
metrics_path = os.path.join(output_dir, "metricas_modelos.csv")
test_metrics_path = os.path.join(output_dir, "metricas_test_interno.csv")
summary_path = os.path.join(output_dir, "resumen_proyecto.json")

best_model.save(model_drive_path)
resultados_val_df.to_csv(metrics_path, index=False)
resultados_test_df.to_csv(test_metrics_path, index=False)

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=4)

print("Modelo final guardado en:", model_drive_path)
print("Métricas de validación guardadas en:", metrics_path)
print("Métricas de test interno guardadas en:", test_metrics_path)
print("Resumen guardado en:", summary_path)

# 11. Predicción en test oficial de Kaggle y `submission.csv`

El test oficial de Kaggle no tiene etiquetas públicas, por eso aquí solo se generan probabilidades para enviar a Kaggle.

In [ ]:
# ===============================================================
# PASO 28: PREDICCIÓN EN TEST OFICIAL Y SUBMISSION
# ===============================================================

test_files = sorted([f for f in os.listdir(test_dir) if f.endswith(".tif")])
test_paths = [os.path.join(test_dir, f) for f in test_files]
test_ids = [os.path.splitext(f)[0] for f in test_files]


def load_tif_test_np(path):
    path = path.decode("utf-8")
    img = Image.open(path).convert("RGB").resize(IMG_SIZE)
    img = np.asarray(img, dtype=np.float32) / 255.0
    return img


def load_tif_test_tf(path):
    img = tf.numpy_function(load_tif_test_np, [path], tf.float32)
    img.set_shape((IMG_SIZE[0], IMG_SIZE[1], 3))
    return img


test_kaggle_ds = (
    tf.data.Dataset.from_tensor_slices(test_paths)
    .map(load_tif_test_tf, num_parallel_calls=AUTOTUNE)
    .batch(BATCH)
    .prefetch(AUTOTUNE)
)

test_kaggle_prob = best_model.predict(test_kaggle_ds, verbose=1).ravel()

submission = pd.DataFrame({
    "id": test_ids,
    "label": test_kaggle_prob,
})

submission_path = os.path.join(output_dir, "submission.csv")
submission.to_csv(submission_path, index=False)

display(submission.head())
print("Submission guardado en:", submission_path)

# 12. Conclusión final

Se desarrolló un pipeline completo de deep learning para detectar cáncer en imágenes histopatológicas.  
Se entrenaron tres arquitecturas diferentes y se seleccionó el mejor modelo según métricas de validación y test interno.  
El modelo final puede apoyar la priorización de imágenes sospechosas, siempre bajo supervisión médica.